# ممیزی کیفیت و صلاحیت دادهٔ Laya برای نیاز املاک

## tl;dr

این بررسی فقط صلاحیت داده را می‌سنجد و هیچ inference، ساخت نمونه یا آموزش انجام نمی‌دهد. پنج batch دستی ۶۸ ردیف یکتا دارند؛ همه synthetic و فاقد صلاحیت real-need و auxiliary fine-tuning هستند. snapshot متوقف‌شدهٔ v3 شامل ۱٬۳۱۲ ردیف و ۱٬۳۰۳ متن یکتا است؛ ۹ متن خروجی تکراری دارد. Gate آموزشی برای هر دو گروه صفر ردیف را می‌پذیرد.

ممیزی تکمیلی 2026-09-28، اختلاف محله را کامل reconcile کرد: ۴۲۱٬۱۰۶ ردیف raw exact به ۴۲۰٬۹۸۹ retained می‌رسد؛ ۱۱۶ ردیف در گروه‌های متعارض و یک ردیف به‌علت category نگاشت‌نشده حذف شده است، بدون mismatch بین crosswalk محله و نگاشت شهر. این هم پوشش نام/آگهی است، نه نیاز واقعی کاربر.

Divar منبعِ آگهی عرضهٔ فروشنده/مشاور است، نه پیامِ نیاز خریدار یا مستأجر. تبدیل آگهی به جملهٔ اول‌شخص، حتی با خواندن دستی، نیت واقعی متقاضی را ایجاد نمی‌کند. کارت رسمی dataset یک‌میلیون آگهی املاک را توصیف می‌کند: [Divar real-estate ads](https://huggingface.co/datasets/divarofficial/real_estate_ads). Laya نیز مدل تصمیم typed است، نه مدل تولید متن: [مستند رسمی Laya](https://github.com/NandhaKishorM/laya).

## Context & Methods

واحد ممیزی یک نمونهٔ خروجی است. از auditهای aggregate-only موجود استفاده می‌شود؛ متن، شناسه‌ها و مقادیر خام نمونه‌ها چاپ نمی‌شوند. آزمون corpus gate برای Laya بررسی می‌کند: متن یکتا، وضعیت synthetic صریح، task type دقیق، typed decisions، provenance، رضایت/مجوز، بازبینی انسانی، patternهای تماس، و منبع عرضه. exit code 1 برای corpusهای غیرمجاز انتظار می‌رود و به معنی شکست اجرای ممیزی نیست.

### Key assumptions

- `sourceRowsRead` در manifest تعداد ثبت‌شده در زمان نرمال‌سازی است؛ coverage script فعلی manifest و catalog را می‌خواند، نه اینکه CSV یک‌میلیونی را دوباره stream کند.
- بررسی phone/email/URL فقط الگوهای محدود را می‌سنجد و clearance جامع PII یا حقوق محتوا نیست.
- پیشنهاد مشتق‌شده از آگهی به‌عنوان ground truth نیاز تلقی نمی‌شود.

## Data

ورودی‌های محلی: `data/divar/manual-derived-need-batch-02..06.jsonl` و `data/divar/divar-hypothetical-needs-laya-full-v3-2026-09-26.jsonl` به‌همراه manifestهایشان. مقایسهٔ پوشش از catalogهای `/post` در `src/data/neighborhoods/catalog` و manifest/crosswalk ثبت‌شده استفاده می‌کند.

## Results

| مجموعه | ردیف | متن یکتا | تکراری | pattern تماس | رضایت | بازبینی انسانی | synthetic | عرضه | واجدشرایط آموزش |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| batchهای دستی Divar | 68 | 68 | 0 | 0 | 0 | 0 | 68 | 0 | 0 |
| snapshot متوقف‌شدهٔ v3 | 1,312 | 1,303 | 9 | 0 | 0 | 0 | 1,312 | 1,312 | 0 |

پوشش منبع نسبت به catalog فعلی: ۱۶/۱۸ دستهٔ ملکی؛ ۳۸۲/۱٬۲۰۷ catalog شهر در crosswalk؛ ۹۲٫۵۷٪ از ردیف‌های نرمال‌شده شهر app-mapped دارند؛ محلهٔ دقیق برای ۴۲٫۱۲٪ ردیف‌ها نگاشت شده است. ۶۸ جفت شهر/محله در crosswalk حل‌نشده‌اند. این اعداد پوششِ نام‌ها/آگهی‌هاست، نه پوشش نیت کاربران.

## Takeaways

هیچ‌کدام از این نمونه‌ها برای آموزش مدل نیاز واقعی قابل‌قبول نیست. تکرارهای v3 از این واقعیت ناشی می‌شوند که dedup منبع قبل از تبدیل انجام شده، ولی یکتایی state نهایی پس از template conversion کنترل نشده است. رفع dedup به‌تنهایی مشکل بنیادی را حل نمی‌کند: آگهی عرضه، هدف نیاز متقاضی نیست. مسیر لازم برای corpus واقعی، پیام نیازِ first-party با opt-in آموزشِ نسخه‌دار و labels تأییدشدهٔ کاربر/بازبین است.

## Reproduce

سلول‌های زیر فقط aggregate auditهای موجود را اجرا می‌کنند؛ فایل ورودی را تغییر نمی‌دهند و مدل را بارگذاری نمی‌کنند. نیازمندی‌ها: Python 3 و Bun؛ مسیرها باید از ریشهٔ repository اجرا شوند.

In [ ]:
import json
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'package.json').is_file():
    raise RuntimeError('Run this notebook from the repository root.')

artifacts = {
    'manual-02': ('data/divar/manual-derived-need-batch-02.jsonl', 'data/divar/manual-derived-need-batch-02.manifest.json'),
    'manual-03': ('data/divar/manual-derived-need-batch-03.jsonl', 'data/divar/manual-derived-need-batch-03.manifest.json'),
    'manual-04': ('data/divar/manual-derived-need-batch-04.jsonl', 'data/divar/manual-derived-need-batch-04.manifest.json'),
    'manual-05': ('data/divar/manual-derived-need-batch-05.jsonl', 'data/divar/manual-derived-need-batch-05.manifest.json'),
    'manual-06': ('data/divar/manual-derived-need-batch-06.jsonl', 'data/divar/manual-derived-need-batch-06.manifest.json'),
    'full-v3-interrupted': ('data/divar/divar-hypothetical-needs-laya-full-v3-2026-09-26.jsonl', 'data/divar/divar-hypothetical-needs-laya-full-v3-2026-09-26.jsonl.manifest.json'),
}

def run_corpus_audit(data_path, manifest_path):
    command = ['bun', 'scripts/datasets/audit-laya-corpus.ts', '--input', data_path, '--manifest', manifest_path]
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, check=False)
    if result.returncode not in (0, 1):
        raise RuntimeError(result.stderr or 'Corpus audit execution failed')
    return json.loads(result.stdout)

audits = {name: run_corpus_audit(*paths) for name, paths in artifacts.items()}
print('All six aggregate corpus audits executed; no source text or row IDs were emitted.')

In [ ]:
columns = ['artifact', 'rows', 'uniqueTexts', 'duplicates', 'contactPatterns', 'consent', 'humanReview', 'synthetic', 'supplySide', 'qualified']
print(' | '.join(columns))
for name, audit in audits.items():
    counts = audit['counts']
    values = [
        name, counts['rows'], counts['uniqueTexts'], counts['duplicateTextRows'],
        counts['piiPatternRows'], counts['withConsentMetadata'], counts['withHumanLabelReview'],
        counts['syntheticRows'], counts['supplySideListingRows'], counts['qualifiedRows'],
    ]
    print(' | '.join(map(str, values)))

manual_rows = sum(audits[name]['counts']['rows'] for name in artifacts if name.startswith('manual-'))
assert manual_rows == 68
assert audits['full-v3-interrupted']['counts']['rows'] == 1312
assert audits['full-v3-interrupted']['counts']['duplicateTextRows'] == 9
assert all(not audit['eligibleForTraining'] for audit in audits.values())
print('Snapshot checks passed: all proposal artifacts are training-ineligible.')

In [ ]:
command = ['bun', 'scripts/datasets/audit-divar-app-coverage.ts']
result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, check=True)
coverage = json.loads(result.stdout)

source = coverage['sourceCityCoverage']
neighborhood = coverage['sourceNeighborhoodCoverage']
categories = coverage['appCategories']
locations = coverage['appLocations']

print(f"Manifested source rows: {coverage['sourceRowsRead']:,}; normalized offer rows: {coverage['normalizedOfferRows']:,}")
print(f"Mapped property categories: {categories['sourceCategoryCount']}/{categories['count']}; absent: {', '.join(categories['missingFromSource'])}")
print(f"App city catalogs represented: {source['sourceCitiesMappedToApp']}/{locations['cityCatalogs']} ({source['shareOfAppCityCatalogsCoveredPct']}%)")
print(f"Offers with mapped app city: {source['shareOfNormalizedOffersWithMappedCityPct']}%; offers with exact mapped neighborhood: {neighborhood['shareOfNormalizedOffersWithMappedNeighborhoodPct']}%")
print(f"App neighborhood IDs: {locations['uniqueCityNeighborhoodIds']:,}; unresolved source city-neighborhood pairs: {neighborhood['unresolvedPairs']}")
print(f"Cloud transfer allowed: {coverage['transferAndRights']['cloudTransferAllowed']}; license review required: {coverage['transferAndRights']['licenseReviewRequiredBeforeRedistribution']}")

assert locations['cityCatalogs'] == 1207
assert locations['uniqueCityNeighborhoodIds'] == 48110
assert categories['missingFromSource'] == ['agency-services', 'land-rent']

## Limitations and next evidence

- Notebook runtime validation is still required in an environment with `jupyter`/`nbclient`; those packages were absent in the audited Python environment. The underlying aggregate commands were executed separately during this audit.
- Docker access was denied, so no current first-party database row count was queried. No claim that historical rows are absent is made.
- Exact app city/neighborhood presence does not prove actual user location intent.
- Divar ODbL metadata is not a complete row-level content-rights, privacy, or downstream fine-tuning determination. Cloud transfer is disallowed by the local source manifest.
- Laya predictions on these derived examples are proxy comparisons, not model accuracy; only independently human-confirmed seeker intent can provide the required evaluation targets.

## تطبیق ردیف‌های محلهٔ دقیق با corpus نگه‌داشته‌شده (2026-09-28)

این ممیزی تکمیلی همان snapshot با checksum ثابت را با قواعد dedup، تعارض دسته/شهر و دسته‌بندیِ خود سازندهٔ corpus بازپخش می‌کند. متن آگهی فقط در حافظه normalize/hash می‌شود؛ خروجی و SQLite موقت، فقط شمارش تجمیعی دارند. این کار اختلاف شمارش را توضیح می‌دهد و دربارهٔ نیت واقعی جویندگان یا صلاحیت حقوقی/حریم خصوصی نتیجه نمی‌دهد.

In [ ]:
import json
import subprocess
import sys

command = [sys.executable, 'scripts/datasets/audit-divar-neighborhood-retained-delta.py']
result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, check=True)
neighborhood_delta = json.loads(result.stdout)
print(json.dumps(neighborhood_delta, ensure_ascii=False, indent=2))

assert neighborhood_delta['crosswalkExactRows'] == 421106
assert neighborhood_delta['exactRowsRetained'] == 420989
assert neighborhood_delta['exactRowsConflictSkipped'] == 116
assert neighborhood_delta['exactRowsWithCategoryConflict'] == 46
assert neighborhood_delta['exactRowsWithCityConflict'] == 84
assert neighborhood_delta['exactRowsWithBothConflicts'] == 14
assert neighborhood_delta['exactRowsUnmappedCategorySkipped'] == 1
assert neighborhood_delta['exactRowsExcluded'] == 117
assert neighborhood_delta.get('exactCrosswalkCityMismatchRows', 0) == 0
assert neighborhood_delta['reconciles'] is True